# LeetCode 1704 — Determine if String Halves Are Alike bằng Python

**Link đề:** https://leetcode.com/problems/determine-if-string-halves-are-alike/
**Độ khó:** Easy · **Chủ đề:** String, Counting

```python
class Solution:
    def halvesAreAlike(self, s: str) -> bool:
        ...
```

## Mục tiêu học
1. Biết cắt chuỗi làm đôi đúng cách: `mid = len(s) // 2`, `s[:mid]`, `s[mid:]`.
2. Kiểm tra nguyên âm có phân biệt hoa/thường: tập `set("aeiouAEIOU")` hoặc `ch.lower() in "aeiou"`.
3. Đếm bằng `sum(...)` trên bool; viết hàm phụ `is_vowel`.
4. Từ "đếm hai nửa rồi so sánh" → **một biến cân bằng** (`+1` nửa trái, `-1` nửa phải) → **hai con trỏ**.
5. Trace, chứng minh, kiểm thử và mở rộng sang các biến thể "so sánh hai nửa".

## 1. Đề bài và ví dụ

Cho chuỗi `s` có **độ dài chẵn**. Chia thành hai nửa bằng nhau: `a` (nửa đầu) và `b` (nửa sau). Hai chuỗi **giống nhau (alike)** nếu có **cùng số nguyên âm** (`'a','e','i','o','u'` và cả chữ hoa `'A','E','I','O','U'`). Trả về `True` nếu `a` và `b` alike.

| `s` | `a` | `b` | nguyên âm a / b | Kết quả |
|---|---|---|---|:---:|
| `"book"` | `"bo"` | `"ok"` | 1 / 1 | `True` |
| `"textbook"` | `"text"` | `"book"` | 1 / 2 | `False` |

Chú ý: `"book"` có `o` hai lần → đếm **2** (đếm số lần xuất hiện, không đếm loại).

**Constraints:** `2 <= len(s) <= 1000`; `len(s)` chẵn; `s` gồm chữ cái hoa và thường.

### 1.1. Tính tay

`"AbCdEfGh"` → `a = "AbCd"` (nguyên âm: A → 1), `b = "EfGh"` (E → 1) ⇒ `True`. Chữ hoa cũng là nguyên âm!

In [1]:
s = "AbCdEfGh"
mid = len(s) // 2
a, b_half = s[:mid], s[mid:]
print("a =", a, "| b =", b_half)
print("nguyên âm trong a:", [ch for ch in a if ch in "aeiouAEIOU"])
print("nguyên âm trong b:", [ch for ch in b_half if ch in "aeiouAEIOU"])

a = AbCd | b = EfGh
nguyên âm trong a: ['A']
nguyên âm trong b: ['E']


## 2. Nền tảng Python cần biết

### 2.1. Cắt đôi chuỗi bằng slicing

`mid = len(s) // 2`. `s[:mid]` lấy chỉ số `0..mid-1`, `s[mid:]` lấy `mid..len-1`. Vì `len(s)` chẵn nên hai nửa dài bằng nhau.

In [2]:
for s in ["book", "textbook", "ab"]:
    mid = len(s) // 2
    print(f"{s!r}: len={len(s)}, mid={mid}, a={s[:mid]!r}, b={s[mid:]!r}")

'book': len=4, mid=2, a='bo', b='ok'
'textbook': len=8, mid=4, a='text', b='book'
'ab': len=2, mid=1, a='a', b='b'


### 2.2. Kiểm tra nguyên âm: chuỗi, `set`, hay `lower()`

Ba cách tương đương:
- `ch in "aeiouAEIOU"` — quét chuỗi 10 ký tự.
- `ch in VOWELS` với `VOWELS = set("aeiouAEIOU")` — tra hash trung bình `O(1)`.
- `ch.lower() in "aeiou"` — đổi về chữ thường rồi kiểm tra 5 ký tự.

`str.lower()` trả về **chuỗi mới** ở dạng chữ thường; chuỗi gốc không đổi.

In [3]:
VOWELS = set("aeiouAEIOU")
for ch in ["a", "E", "b", "U", "y"]:
    print(ch, "| in str:", ch in "aeiouAEIOU", "| in set:", ch in VOWELS, "| lower:", ch.lower() in "aeiou")

a | in str: True | in set: True | lower: True
E | in str: True | in set: True | lower: True
b | in str: False | in set: False | lower: False
U | in str: True | in set: True | lower: True
y | in str: False | in set: False | lower: False


### 2.3. Hàm phụ `is_vowel` và `sum` trên bool

Đặt điều kiện vào hàm có tên rõ nghĩa giúp code dễ đọc (gợi ý của đề cũng khuyên vậy). `True` được cộng như `1`.

In [4]:
def is_vowel(ch: str) -> bool:
    return ch in VOWELS

word = "textbook"
print([is_vowel(ch) for ch in word])
print("số nguyên âm:", sum(is_vowel(ch) for ch in word))

[False, True, False, False, False, True, True, False]
số nguyên âm: 3


### 2.4. `enumerate` — vừa có chỉ số vừa có ký tự

Ta cần biết ký tự đang ở **nửa nào** → cần chỉ số `i`: `i < mid` là nửa trái.

In [5]:
s = "book"
mid = len(s) // 2
for i, ch in enumerate(s):
    side = "trái" if i < mid else "phải"
    print(i, ch, side)

0 b trái
1 o trái
2 o phải
3 k phải


## 3. Cách trực giác — đếm hai nửa rồi so sánh

1. Cắt `a = s[:mid]`, `b = s[mid:]`.
2. Đếm nguyên âm của `a`, của `b`.
3. Trả `count_a == count_b`.

In [6]:
from typing import List

def halves_alike_two_counts(s: str) -> bool:
    mid = len(s) // 2
    a, b_ = s[:mid], s[mid:]
    count_a = 0
    for ch in a:
        if ch in VOWELS:
            count_a += 1
    count_b = 0
    for ch in b_:
        if ch in VOWELS:
            count_b += 1
    return count_a == count_b

print(halves_alike_two_counts("book"), halves_alike_two_counts("textbook"))   # True False

True False


### Chi phí và chỗ có thể gọn hơn

Time `O(n)`, Space `O(n)` vì slicing tạo hai chuỗi mới dài `n/2`. Đây đã là lời giải tốt — `n ≤ 1000`. Nhưng ta có hai vòng lặp gần giống nhau (lặp code) và bộ nhớ phụ do cắt chuỗi. Có thể:
- **Gom hai vòng làm một**: chỉ cần **hiệu** `count_a - count_b`, không cần hai số riêng.
- **Không cắt chuỗi**: dùng chỉ số để biết nửa nào.

## 4. Insight — chỉ cần một biến "cân bằng"

`count_a == count_b` ⇔ `count_a - count_b == 0`. Duyệt một lượt:
- gặp nguyên âm ở **nửa trái** → `balance += 1`;
- gặp nguyên âm ở **nửa phải** → `balance -= 1`.

Cuối cùng `balance == 0` ⇔ alike. Đây là kỹ thuật **"cân bằng / hiệu số"** dùng rất nhiều (ngoặc hợp lệ, số 0 và 1 bằng nhau…).

In [7]:
s = "textbook"
mid = len(s) // 2
balance = 0
for i, ch in enumerate(s):
    if ch in VOWELS:
        balance += 1 if i < mid else -1
print("balance =", balance, "-> alike?", balance == 0)

balance = -1 -> alike? False


## 5. Lời giải chính — một lượt duyệt với `balance`

**Đoạn 1:** `mid = len(s) // 2`, `balance = 0`.
**Đoạn 2:** duyệt `i` từ `0` đến `mid - 1`, mỗi bước xét **cặp** `s[i]` (trái) và `s[i + mid]` (phải): cộng 1 nếu `s[i]` là nguyên âm, trừ 1 nếu `s[i + mid]` là nguyên âm.
**Đoạn 3:** trả `balance == 0`.

Cách duyệt cặp này đi `n/2` vòng, không cần `if i < mid`.

In [8]:
def halves_alike_balance(s: str) -> bool:
    mid = len(s) // 2
    balance = 0
    for i in range(mid):
        if s[i] in VOWELS:
            balance += 1
        if s[i + mid] in VOWELS:
            balance -= 1
    return balance == 0

for s in ["book", "textbook", "AbCdEfGh", "MerryChristmas"]:
    print(s, "->", halves_alike_balance(s))

book -> True
textbook -> False
AbCdEfGh -> True
MerryChristmas -> False


## 6. TRACE từng bước

Ví dụ `s = "textbook"`, `mid = 4`. Cặp `(i, i + mid)`:

| i | s[i] | nguyên âm? | s[i+4] | nguyên âm? | balance sau |
|---:|:---:|:---:|:---:|:---:|---:|
| 0 | t | ✗ | b | ✗ | 0 |
| 1 | e | ✓ (+1) | o | ✓ (−1) | 0 |
| 2 | x | ✗ | o | ✓ (−1) | −1 |
| 3 | t | ✗ | k | ✗ | −1 |

`balance = −1 ≠ 0` ⇒ `False`. Dấu âm cho biết **nửa phải nhiều nguyên âm hơn** 1 cái.

In [9]:
def trace_halves(s: str) -> bool:
    mid = len(s) // 2
    balance = 0
    print(f"s = {s!r}, mid = {mid}")
    for i in range(mid):
        left, right = s[i], s[i + mid]
        dl = 1 if left in VOWELS else 0
        dr = -1 if right in VOWELS else 0
        balance += dl + dr
        print(f"  i={i}: trái {left!r} ({'+1' if dl else ' 0'}), phải {right!r} ({'-1' if dr else ' 0'}) -> balance = {balance}")
    print("  kết quả:", balance == 0)
    return balance == 0

trace_halves("textbook")
trace_halves("book")

s = 'textbook', mid = 4
  i=0: trái 't' ( 0), phải 'b' ( 0) -> balance = 0
  i=1: trái 'e' (+1), phải 'o' (-1) -> balance = 0
  i=2: trái 'x' ( 0), phải 'o' (-1) -> balance = -1
  i=3: trái 't' ( 0), phải 'k' ( 0) -> balance = -1
  kết quả: False
s = 'book', mid = 2
  i=0: trái 'b' ( 0), phải 'o' (-1) -> balance = -1
  i=1: trái 'o' (+1), phải 'k' ( 0) -> balance = 0
  kết quả: True


True

## 7. Các lời giải khác

### 7.1. Một dòng với `sum` + slicing

In [10]:
def halves_alike_oneliner(s: str) -> bool:
    mid = len(s) // 2
    return sum(ch in VOWELS for ch in s[:mid]) == sum(ch in VOWELS for ch in s[mid:])

print(halves_alike_oneliner("book"), halves_alike_oneliner("textbook"))

True False


### 7.2. `lower()` toàn chuỗi rồi `str.count`

Đổi cả chuỗi về chữ thường một lần; sau đó với mỗi nguyên âm `v`, `half.count(v)` đếm số lần xuất hiện. `str.count(sub)` đếm số lần chuỗi con xuất hiện (không chồng lấn).

In [11]:
def halves_alike_count(s: str) -> bool:
    t = s.lower()
    mid = len(t) // 2
    a, b_ = t[:mid], t[mid:]
    return sum(a.count(v) for v in "aeiou") == sum(b_.count(v) for v in "aeiou")

print("'textbook'.count('o') =", "textbook".count("o"))
print(halves_alike_count("book"), halves_alike_count("textbook"), halves_alike_count("AbCdEfGh"))

'textbook'.count('o') = 2
True False True


### 7.3. Hai con trỏ từ hai đầu

Con trỏ `i` chạy từ đầu (nửa trái), `j` chạy từ cuối (nửa phải), đến khi gặp nhau. Mỗi nửa vẫn được duyệt đủ một lần — thứ tự không ảnh hưởng tới **số lượng**.

In [12]:
def halves_alike_two_pointers(s: str) -> bool:
    i, j = 0, len(s) - 1
    balance = 0
    while i < j:
        balance += (s[i] in VOWELS) - (s[j] in VOWELS)   # bool - bool -> -1, 0, 1
        i += 1
        j -= 1
    return balance == 0

print(True - False, False - True, True - True)
print(halves_alike_two_pointers("book"), halves_alike_two_pointers("textbook"))

1 -1 0
True False


### 7.4. Bảng so sánh

| Cách | Ý tưởng | Time | Space phụ | Ghi chú |
|---|---|---|---|---|
| Hai lần đếm | cắt hai nửa, hai vòng | `O(n)` | `O(n)` | trực giác |
| `sum` một dòng | như trên nhưng gọn | `O(n)` | `O(n)` | rất Pythonic |
| `lower` + `count` | 5 lần `count` mỗi nửa | `O(n)` (hằng số ×5) | `O(n)` | dùng API chuỗi |
| **Balance theo cặp** | `+1` trái, `−1` phải | `O(n)` | `O(1)` | khuyên dùng |
| Hai con trỏ | `i` đầu, `j` cuối | `O(n)` | `O(1)` | luyện kỹ thuật con trỏ |

`n = len(s)`. Tập `VOWELS` có 10 phần tử cố định ⇒ `O(1)`.

## 8. Code nộp LeetCode

In [13]:
class Solution:
    def halvesAreAlike(self, s: str) -> bool:
        vowels = set("aeiouAEIOU")
        mid = len(s) // 2
        balance = 0
        for i in range(mid):
            if s[i] in vowels:
                balance += 1
            if s[i + mid] in vowels:
                balance -= 1
        return balance == 0

sol = Solution()
print(sol.halvesAreAlike("book"), sol.halvesAreAlike("textbook"))

True False


Bản ngắn gọn (dễ nhớ khi thi):

In [14]:
class SolutionShort:
    def halvesAreAlike(self, s: str) -> bool:
        v = set("aeiouAEIOU")
        mid = len(s) // 2
        return sum(c in v for c in s[:mid]) == sum(c in v for c in s[mid:])

print(SolutionShort().halvesAreAlike("book"), SolutionShort().halvesAreAlike("textbook"))

True False


## 9. Vì sao đúng?

Gọi `A` = số nguyên âm của nửa trái, `B` = của nửa phải.

**Invariant của vòng balance:** sau khi xét các cặp `i = 0..k-1`, `balance = A_k − B_k` với `A_k` = số nguyên âm trong `s[0..k-1]`, `B_k` = số nguyên âm trong `s[mid..mid+k-1]`.
- Ban đầu `k = 0`: `balance = 0 = 0 − 0`.
- Bước `k`: `s[k]` (trái) làm `A` tăng 1 nếu là nguyên âm ⇒ `balance += 1`; `s[k + mid]` (phải) làm `B` tăng 1 ⇒ `balance -= 1`. Invariant giữ.
- Khi `k = mid`: hai đoạn đã phủ đúng toàn bộ `a = s[0..mid-1]` và `b = s[mid..n-1]` (vì `n = 2·mid`). Vậy `balance = A − B`, và `balance == 0 ⇔ A == B`.

**Hai con trỏ:** `i` đi qua `0..mid-1`, `j` đi qua `n-1..mid` — cũng phủ đúng hai nửa (`i < j` dừng đúng lúc `i = mid`, `j = mid - 1`). Phép cộng giao hoán nên thứ tự duyệt không ảnh hưởng.

## 10. Độ phức tạp

`n = len(s)`.
- **Time:** `O(n)` — mỗi ký tự được kiểm tra đúng một lần; mỗi lần tra `set` cỡ 10 là `O(1)`.
- **Space:** `O(1)` cho cách balance / hai con trỏ (chỉ vài biến và tập 10 phần tử). Cách slicing tốn `O(n)` cho hai nửa.
- Không thể nhanh hơn `O(n)`: phải xem mọi ký tự (một ký tự chưa xem có thể thay đổi kết quả).

## 11. Bộ test

1. Ví dụ đề: `"book"` → `True`, `"textbook"` → `False`.
2. Biên: độ dài 2 (`"aa"`, `"ab"`, `"bb"`), không có nguyên âm (`"bcdf"` → `True` vì 0 == 0).
3. Dễ sai: **chữ hoa** (`"Aa"` → True, `"Ab"` → False), nguyên âm lặp, nguyên âm nằm sát ranh giới `mid`.
4. Đối chiếu 7 cài đặt trên chuỗi ngẫu nhiên hoa/thường.

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

S = Solution()
check("đề 1", S.halvesAreAlike("book"), True)
check("đề 2", S.halvesAreAlike("textbook"), False)
check("len 2 cùng nguyên âm", S.halvesAreAlike("aa"), True)
check("len 2 lệch", S.halvesAreAlike("ab"), False)
check("không nguyên âm", S.halvesAreAlike("bcdf"), True)
check("chữ hoa Aa", S.halvesAreAlike("Aa"), True)
check("chữ hoa Ab", S.halvesAreAlike("Ab"), False)
check("hoa thường lẫn", S.halvesAreAlike("AbCdEfGh"), True)
check("sát ranh giới", S.halvesAreAlike("bbaEbb"), True)
check("MerryChristmas", S.halvesAreAlike("MerryChristmas"), False)

PASS đề 1: True
PASS đề 2: False
PASS len 2 cùng nguyên âm: True
PASS len 2 lệch: False
PASS không nguyên âm: True
PASS chữ hoa Aa: True
PASS chữ hoa Ab: False
PASS hoa thường lẫn: True
PASS sát ranh giới: True
PASS MerryChristmas: False


In [16]:
import random, string

impls = [halves_alike_two_counts, halves_alike_balance, halves_alike_oneliner,
         halves_alike_count, halves_alike_two_pointers,
         Solution().halvesAreAlike, SolutionShort().halvesAreAlike]
random.seed(1704)
letters = "aeiouAEIOUbcdBCDxyzXYZ"
for _ in range(3000):
    n = 2 * random.randint(1, 20)
    s = "".join(random.choice(letters) for _ in range(n))
    res = {f(s) for f in impls}
    assert len(res) == 1, (s, res)
print("Đối chiếu 7 cài đặt trên 3000 chuỗi ngẫu nhiên: PASS")

Đối chiếu 7 cài đặt trên 3000 chuỗi ngẫu nhiên: PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1 — Quên nguyên âm chữ hoa

In [17]:
def wrong_lowercase_only(s):
    mid = len(s) // 2
    v = set("aeiou")                         # SAI: thiếu AEIOU
    return sum(c in v for c in s[:mid]) == sum(c in v for c in s[mid:])

print("sai:", wrong_lowercase_only("Ab"), "| đúng:", halves_alike_balance("Ab"))

sai: True | đúng: False


**Sửa:** dùng `set("aeiouAEIOU")` hoặc `c.lower() in "aeiou"`.

### Lỗi 2 — Đếm **loại** nguyên âm thay vì **số lần**

Dùng `set(a) & vowels` sẽ chỉ biết *có những nguyên âm nào*, không biết mỗi cái xuất hiện bao nhiêu lần.

In [18]:
def wrong_distinct(s):
    mid = len(s) // 2
    v = set("aeiouAEIOU")
    return len(set(s[:mid]) & v) == len(set(s[mid:]) & v)   # SAI: 'oo' chỉ tính 1

print("sai:", wrong_distinct("textbook"), "| đúng:", halves_alike_balance("textbook"))

sai: True | đúng: False


`set` và phép giao `&`: `set(a) & v` là các phần tử **chung** của hai tập. Hữu ích, nhưng không phải để đếm số lần. **Sửa:** đếm từng ký tự bằng `sum(...)`.

### Lỗi 3 — Cắt sai điểm giữa (`/` thay vì `//`, hoặc lệch 1)

In [19]:
s = "textbook"
try:
    print(s[:len(s) / 2])
except TypeError as e:
    print("TypeError:", e)
mid_wrong = len(s) // 2 + 1
print("lệch 1:", s[:mid_wrong], "|", s[mid_wrong:])

TypeError: slice indices must be integers or None or have an __index__ method
lệch 1: textb | ook


`len(s) / 2` là `float` (`4.0`), không dùng làm chỉ số được. **Sửa:** `mid = len(s) // 2`; dùng `s[:mid]` và `s[mid:]` (hai lát cắt chung điểm `mid`, không bỏ sót, không trùng).

### Lỗi 4 — So sánh nguyên âm bằng chuỗi `or` dài và sai cú pháp

`if ch == 'a' or 'e' or 'i':` **luôn đúng** vì `'e'` là chuỗi khác rỗng (truthy).

In [20]:
ch = "z"
print("ch == 'a' or 'e' ->", ch == 'a' or 'e')        # trả về 'e' (truthy) -> if luôn đúng
print("bool(...) ->", bool(ch == 'a' or 'e'))
print("cách đúng ->", ch in "aeiouAEIOU")

ch == 'a' or 'e' -> e
bool(...) -> True
cách đúng -> False


**Sửa:** dùng `ch in VOWELS` (hoặc viết đủ `ch == 'a' or ch == 'e' or ...`).

## 13. Mẹo nhận dạng pattern

- "So sánh **số lượng** của hai phần" → dùng **một biến hiệu số** (`+1`/`−1`), kiểm tra `== 0`.
- Hai nửa bằng nhau → duyệt **cặp** `(i, i + mid)` hoặc **hai con trỏ** hai đầu.
- Kiểm tra thuộc nhóm ký tự cố định → `set` hằng (định nghĩa một lần), nhớ cả chữ hoa/thường.
- Bài tương tự: 1221 Split a String in Balanced Strings (biến balance), 1832 Check if the Sentence Is Pangram (tập ký tự), 1347 Minimum Number of Steps to Make Two Strings Anagram (đếm hai phía), 125 Valid Palindrome (hai con trỏ).

## 14. Biến thể

### Biến thể 1 — Độ dài lẻ: bỏ ký tự giữa

In [21]:
def alike_odd(s: str) -> bool:
    mid = len(s) // 2
    left = s[:mid]
    right = s[mid + 1:] if len(s) % 2 else s[mid:]
    return sum(c in VOWELS for c in left) == sum(c in VOWELS for c in right)

print(alike_odd("abXba"), alike_odd("aXb"), alike_odd("book"))

True False True


### Biến thể 2 — Hai nửa có cùng **multiset** nguyên âm (cùng loại, cùng số lần)

Dùng `Counter` chỉ trên nguyên âm (đổi về chữ thường). Hai `Counter` so sánh bằng `==`.

In [22]:
from collections import Counter

def same_vowel_multiset(s: str) -> bool:
    mid = len(s) // 2
    ca = Counter(c.lower() for c in s[:mid] if c in VOWELS)
    cb = Counter(c.lower() for c in s[mid:] if c in VOWELS)
    return ca == cb

print(same_vowel_multiset("bOok"), same_vowel_multiset("baeb"), same_vowel_multiset("abba"))

True False True


### Biến thể 3 — Tìm vị trí cắt `k` (1 ≤ k < n) sao cho hai phần có cùng số nguyên âm

Dùng **tổng tiền tố** (prefix sum): `pre[k]` = số nguyên âm trong `s[:k]`. Phần phải có `total - pre[k]`. Cần `pre[k] == total - pre[k]`.

In [23]:
def balanced_cuts(s: str) -> list:
    pre = [0]
    for c in s:
        pre.append(pre[-1] + (c in VOWELS))
    total = pre[-1]
    return [k for k in range(1, len(s)) if 2 * pre[k] == total]

print(balanced_cuts("textbook"), balanced_cuts("aXa"), balanced_cuts("abc"))

[] [1, 2] []


## 15. Bài tập (tăng dần)

> Hàm bài tập chỉ có `pass`, **không được gọi** khi chạy notebook.

### Bài 1 — Dự đoán output
Không chạy code: `"AeIoUb"`, `"UuUuUu"`, `"zzzzZZ"` cho kết quả gì?

In [24]:
def exercise_1():
    # TODO: trả về tuple 3 giá trị bool bạn dự đoán
    pass

# assert exercise_1() == tuple(Solution().halvesAreAlike(x) for x in ["AeIoUb", "UuUuUu", "zzzzZZ"])

### Bài 2 — Viết `count_vowels(t)` không dùng `sum`

In [25]:
def count_vowels(t: str) -> int:
    # TODO: vòng for + biến đếm
    pass

# assert count_vowels("textBOOK") == 3

### Bài 3 — Điền chỗ trống phiên bản balance

In [26]:
def balance_fill(s: str) -> bool:
    mid = len(s) // 2
    balance = 0
    # TODO 1: duyệt i trong range(mid)
    # TODO 2: +1 nếu s[i] là nguyên âm; -1 nếu s[i + mid] là nguyên âm
    # TODO 3: trả balance == 0
    pass

# assert balance_fill("book") == True and balance_fill("textbook") == False

### Bài 4 — Trả về chênh lệch có dấu
Viết `vowel_diff(s)` trả về `A − B` (nửa trái trừ nửa phải).

In [27]:
def vowel_diff(s: str) -> int:
    # TODO
    pass

# assert vowel_diff("textbook") == -1 and vowel_diff("book") == 0 and vowel_diff("aabb") == 2

### Bài 5 — Biến thể: so sánh số **phụ âm** (chữ cái không phải nguyên âm)

In [28]:
def consonant_halves_alike(s: str) -> bool:
    # TODO: lưu ý s chỉ gồm chữ cái
    pass

# assert consonant_halves_alike("book") == True and consonant_halves_alike("textbook") == False

### Bài 6 — Mini challenge: số thay đổi ít nhất
Mỗi thao tác được đổi **một ký tự** thành bất kỳ chữ cái nào. Cần ít nhất bao nhiêu thao tác để hai nửa alike?

In [29]:
def min_changes_to_alike(s: str) -> int:
    # TODO: nghĩ xem một thao tác thay đổi |A - B| tối đa bao nhiêu
    pass

# assert min_changes_to_alike("textbook") == 1 and min_changes_to_alike("aabb") == 2 and min_changes_to_alike("book") == 0

## 16. Gợi ý 3 mức

**Bài 1:** (1) Đếm từng nửa, nhớ chữ hoa. (2) `"AeI"` có 3, `"oUb"` có 2. (3) `False, True, True`.

**Bài 2:** (1) Biến `cnt = 0`. (2) `for c in t: if c in VOWELS: cnt += 1`. (3) `return cnt`.

**Bài 3:** (1) Như mục 5. (2) Hai câu `if` độc lập (không `elif`!). (3) `return balance == 0`.

**Bài 4:** (1) Chính là `balance`. (2) Trả `balance` thay vì so sánh. (3) `"aabb"`: trái 2, phải 0 → 2.

**Bài 5:** (1) Phụ âm = chữ cái không thuộc `VOWELS`. (2) Hai nửa dài bằng nhau ⇒ phụ âm bằng nhau ⇔ nguyên âm bằng nhau! (3) Có thể `return Solution().halvesAreAlike(s)`.

**Bài 6:** (1) Một thao tác đổi một ký tự ở một nửa. (2) Mỗi thao tác làm `A − B` thay đổi tối đa 1. (3) Đáp án `abs(A − B)` (luôn làm được vì đổi nguyên âm ↔ phụ âm ở nửa nhiều hơn).

## 17. Đáp án tham khảo

In [30]:
def ans_exercise_1():
    return (False, True, True)

def ans_count_vowels(t):
    cnt = 0
    for c in t:
        if c in VOWELS:
            cnt += 1
    return cnt

def ans_balance_fill(s):
    mid = len(s) // 2
    balance = 0
    for i in range(mid):
        if s[i] in VOWELS:
            balance += 1
        if s[i + mid] in VOWELS:
            balance -= 1
    return balance == 0

def ans_vowel_diff(s):
    mid = len(s) // 2
    return ans_count_vowels(s[:mid]) - ans_count_vowels(s[mid:])

def ans_consonant_halves_alike(s):
    mid = len(s) // 2
    return sum(c not in VOWELS for c in s[:mid]) == sum(c not in VOWELS for c in s[mid:])

def ans_min_changes_to_alike(s):
    return abs(ans_vowel_diff(s))

assert ans_exercise_1() == tuple(Solution().halvesAreAlike(x) for x in ["AeIoUb", "UuUuUu", "zzzzZZ"])
assert ans_count_vowels("textBOOK") == 3
assert ans_balance_fill("book") == True and ans_balance_fill("textbook") == False
assert ans_vowel_diff("textbook") == -1 and ans_vowel_diff("book") == 0 and ans_vowel_diff("aabb") == 2
assert ans_consonant_halves_alike("book") == True and ans_consonant_halves_alike("textbook") == False
assert ans_min_changes_to_alike("textbook") == 1 and ans_min_changes_to_alike("aabb") == 2 and ans_min_changes_to_alike("book") == 0
for _ in range(500):
    s = "".join(random.choice(letters) for _ in range(2 * random.randint(1, 10)))
    assert ans_consonant_halves_alike(s) == Solution().halvesAreAlike(s)
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def halvesAreAlike(self, s: str) -> bool:
        v = set("aeiouAEIOU")          # nhớ chữ hoa!
        mid = len(s) // 2               # // chứ không phải /
        balance = 0
        for i in range(mid):            # cặp (trái i, phải i + mid)
            balance += (s[i] in v) - (s[i + mid] in v)
        return balance == 0
```

**Khung tư duy:** "so sánh số lượng hai phần" → **một biến hiệu số** → duyệt một lượt → `== 0`.

**Checklist trước khi submit**
- [ ] Đã tính cả nguyên âm **chữ hoa**?
- [ ] `mid = len(s) // 2`, cắt `s[:mid]` / `s[mid:]` (không lệch 1)?
- [ ] Đếm **số lần** xuất hiện, không đếm loại?
- [ ] Hai câu `if` độc lập cho trái và phải (không dùng `elif`)?
- [ ] Đã thử chuỗi không có nguyên âm và độ dài 2?
- [ ] Tên method `halvesAreAlike`, trả về `bool`?